### Installing required libraries

This cell installs the Python packages needed by the notebook to read and process Excel files.

In [ ]:
# %pip install pybis
%pip install pandas
%pip install openpyxl

In [ ]:
import pandas as pd#we import pandas to read the excel file and convert it into a dataframe
import re#will be used for parsing
from openpyxl import load_workbook#will be used to edit the excel file and add formatting
from openpyxl.styles import Font, PatternFill#will be used to edit the excel file and add formatting

### Connecting to openBIS

This section is currently under construction. It describes how to log in with an openBIS session token: sign in to the Bam Data Store website, open the **Tools** tab, select **User Profile**, and copy the session token for use in the next cell.

In [ ]:
# o = Openbis("https://playground.datastore.bam.de/")

# o.set_token("")

In [ ]:
# # print(o.get_spaces())#please uncomment this line and execute it to see all the spaces and find the code for yours in the table displayed

# myspace = o.get_space('')#input your space's code here to save it in the myspace variable

In [ ]:
# myspace.get_projects()#to see current projects in the space

In [ ]:
# project = myspace.get_project('PARSER_TEST')
# project.attrs.all()#to see all the attributes of the project

In [ ]:
# project.get_experiments()#to see all the experiments in the project

In [ ]:
# collection = project.get_collections()[0]

In [ ]:
# collection.attrs.all()#to see all the attributes of the collection

### Selecting the input file

Enter the path to the sigmaBAM-exported Excel file in the next cell. The parser uses this path to load the source data.

In [ ]:
file_path = 'datasets\input\example_01_simplified.xlsx'


### Reading the input Excel file

The selected Excel file is loaded into a pandas dataframe so its values can be inspected and transformed.

In [ ]:
df = pd.read_excel(file_path,header=0, engine='openpyxl', dtype=str)
df#print the dataframe to see the data in it

### Preparing the output dataframe

An empty pandas dataframe is created to hold the parsed values and the columns required by openBIS.

In [ ]:
out_df = pd.DataFrame()

### Inspecting the source columns

This cell displays the column names in the original dataframe to verify which source fields are available for parsing.

In [ ]:
df.columns

### Defining column mappings and product category codes

This cell maps source column names to their corresponding output names and defines the allowed product category codes used during parsing.

In [ ]:
MAPPING_COLUMNS = {
    "Handelsname": "Name",
    "Synonyme": "Alternative Name",
    "CAS-Nr": "CAS Registry Number",
    "Hersteller": "Manufacturer",
    "Konzentration [%]": "Concentration",
    "Dichte [g/cm\u00b3]": "Density",
}

allowed_pc_codes = {"PC0", "PC1", "PC2", "PC3", "PC4", "PC7", "PC8", "PC9A", "PC9B", "PC9C",
    "PC11", "PC12", "PC13", "PC14", "PC15", "PC16", "PC17", "PC18", "PC20", "PC21",
    "PC23", "PC24", "PC25", "PC26", "PC27", "PC28", "PC29", "PC30", "PC31", "PC32",
    "PC33", "PC34", "PC35", "PC36", "PC37", "PC38", "PC39", "PC40", "PC41", "PC42"}

### Validating required source columns

This cell checks that every source column needed by the mapping exists in the original dataframe. A `ValueError` is raised when a required column is missing.

In [ ]:
for x in MAPPING_COLUMNS.keys():
    if x not in df.columns:
        raise ValueError(f"Column '{x}' is missing from the DataFrame.")

### Creating the `$` identifier column

The `Umgang-Id` value from the source dataframe is changed to `$` and stored in the output dataframe.

In [ ]:
out_df['$'] = df['Umgang-Id'].apply(lambda x: f"${x}")
out_df

### Mapping source columns to the output

Each source column listed in `MAPPING_COLUMNS` is copied into the output dataframe under its mapped name, such as `Synonyme` becoming `Alternative Name`.

In [ ]:
for source, target in MAPPING_COLUMNS.items():
    out_df[target] = df[source]
out_df

### Defining the final output columns

This cell lists all mandatory columns expected in the final openBIS-compatible dataframe. Columns without source values will be filled with `None`.

In [ ]:
final_columns = [
    '$', 'Name', 'Alternative Name', 'IUPAC Name', 'CAS Registry Number', 'Manufacturer',
    'Supplier', 'Lot/Batch Number', 'External Barcode', 'Description', 'Product Category',
    'Hazardous Substance', 'BAM Organizational Entity', 'Complete BAM Location',
    'Responsible person', 'Molar Mass', 'Density', 'Concentration',
    'Bottling Date', 'Opening Date', 'Expiration Date', 'Empty', 'Notes', 'Comments', 'Code', 'Identifier'
]

### Completing the output columns

Each column in `final_columns` is added to the output dataframe if it does not already exist, with missing values initialized to `None`.

In [ ]:
for col in final_columns:
    if col not in out_df.columns:
        out_df[col] = None

In [ ]:
display(out_df)#displaying the output dataframe to see the columns and data in it

### Determining hazardous substance status

This cell checks whether any hazard-related source column contains a value. If at least one does, the output `Hazardous Substance` column is set to `true`; otherwise it is set to `false`.

In [ ]:
hazard_cols = ["H-Sätze", "EUH-Sätze", "P-Sätze", "CMR"]

has_hazard = any(
    df[col].notna().any()
    for col in hazard_cols
    if col in df.columns
)
out_df['Hazardous Substance'] = 'true' if has_hazard else 'false'

### Checking the hazardous substance values

This cell displays the resulting `Hazardous Substance` column so the calculated status can be reviewed.

In [ ]:
out_df['Hazardous Substance']

### Determining the product category

This cell checks whether `Produktkategorie` contains a value, extracts its first code, and copies it to `Product Category` when the code is in the allowed list.

In [ ]:
if df['Produktkategorie'].any():#filling product category
    words = list(df['Produktkategorie'].str.split())
    if str(words[0][0]) in allowed_pc_codes:
        out_df["Product Category"] = words[0][0]
out_df['Product Category']

### Building the BAM location

This cell combines the available values from `Liegenschaft`, `Haus`, `Etage`, and `Raum-Nr` using `/` as a separator, then stores the resulting path in `Complete BAM Location`.

In [ ]:
location_cols = ["Liegenschaft", "Haus", "Etage", "Raum-Nr"]
loc_string = ""
for col in location_cols:
    if not pd.isna(df[col].iloc[0]):
        loc_string += str(df[col].iloc[0]) + "/"
        # print("yes")
    # print(df[col].iloc[0])
if len(loc_string) > 0:
    loc_string = loc_string.rstrip("/")
print(loc_string)
out_df['Complete BAM Location'] = loc_string

### Checking the BAM location

This cell displays the generated `Complete BAM Location` value for review.

In [ ]:
out_df['Complete BAM Location']

### Creating the chemical code

This cell creates the output `Code` by combining the `CHEM` prefix, the organizational unit, and the zero-padded `Umgang-Id` value.

In [ ]:
umgang_id = str(df[ "Umgang-Id" ][0]).zfill(4)
entity = df['Organisationseinheit'][0]
if pd.isna(entity) or pd.isna(umgang_id):
    out_df['Code'] = None
else:
    out_df['Code'] = f"CHEM-{entity}-{umgang_id}"
out_df['Code']

### Creating the identifier

This cell assigns an openBIS identifier to the output dataframe. The current value is entered manually and is intended to be automated later.

In [ ]:
out_df['Identifier'] = '/VP.1_NARSHAD/PARSER_TEST/CHEM-5.4-0001' 
out_df['Identifier']

### Creating the responsible person

This cell converts the `AntragstellerIn` name into the corresponding BAM person path. German umlauts are transliterated, and incomplete or missing names result in `None`.

In [ ]:
if 'AntragstellerIn' in df.columns:
    name = df['AntragstellerIn'][0]
    if not isinstance(name, str) or name.strip() == "":
        out_df['Responsible person'] = None
    else:
        replacements = {'ä': 'ae', 'ö': 'oe', 'ü': 'ue', 'Ä': 'Ae', 'Ö': 'Oe', 'Ü': 'Ue'}
        for orig, repl in replacements.items():
            name = name.replace(orig, repl)
        parts = name.strip().split()
        if len(parts) < 2:
            out_df['Responsible person'] = None
        else:
            first_letter = parts[-1][0].upper()
            last_part = parts[0][:7].upper()
            out_df['Responsible person'] = f"/BAM_GLOBAL/BAM_DATA/{first_letter}{last_part}"
out_df['Responsible person']

### Defining the BAM divisions

This cell provides the list of divisions used to translate an organizational unit code into its full BAM organizational entity name.

In [ ]:
Division_list = [
    "1.1 Inorganic Trace Analysis",
    "1.2 Biophotonics",
    "1.3 Instrumental Analytics",
    "1.4 Process Analytical Technology",
    "1.5 Protein Analysis",
    "1.6 Inorganic Reference Materials",
    "1.7 Organic Trace and Food Analysis",
    "1.8 Environmental Analysis",
    "1.9 Chemical and Optical Sensing",
    "2.1 Safety of Energy Carriers",
    "2.2 Process Simulation",
    "2.3 Classification of Hazardous Substances and Dangerous Goods",
    "2.4 Testing and Evaluation of Explosives and Pyrotechnics",
    "2.5 Conformity Assessment Explosives and Pyrotechnics",
    "3.1 Safety of Dangerous Goods Packagings and Batteries",
    "3.2 Safety of Energy Storage Systems",
    "3.3 Safety of Transport Containers",
    "3.4 Safety of Storage Containers",
    "3.5 Safety of Gas Storage Systems and Tanks For Dangerous Goods",
    "3.6 Electrochemical Energy Materials",
    "4.1 Biodeterioration and Reference Organisms",
    "4.2 Material-Microbiome Interactions",
    "4.3 Molecular and Applied Entomology",
    "4.4 Thermochemical Residues Treatment and Resource Recovery",
    "4.5 Analysis of Artefacts and Cultural Assets",
    "5.1 Microstructural Design and Degradation",
    "5.2 Metallic High-Temperature Materials",
    "5.3 Polymer Matrix Composites",
    "5.4 Advanced Multi-materials Processing",
    "5.5 Materials Modelling",
    "5.6 Glasses",
    "6.1 Surface and Thin Film Analysis",
    "6.2 Material and Surface Technologies",
    "6.3 Structure Analysis",
    "6.4 Materials Informatics",
    "6.5 Synthesis and Scattering of Nanostructured Materials",
    "6.6 Digital Materials Chemistry",
    "6.7 Materials Synthesis and Design",
    "7.1 Building Materials",
    "7.2 Buildings and Structures",
    "7.3 Fire Engineering",
    "7.4 Technology of Construction Materials",
    "7.5 Technical Properties of Polymeric Materials",
    "7.6 Corrosion and Corrosion Protection",
    "7.7 Modelling and Simulation",
    "8.1 Sensors, Measurement and Testing Methods",
    "8.2 Non-Destructive Testing Methods For Civil Engineering",
    "8.3 Thermographic Methods",
    "8.4 Acoustic and Electromagnetic Methods",
    "8.5 X-Ray Imaging",
    "8.6 Fibre Optic Sensors",
    "9.1 Components For Energy Carriers",
    "9.2 Testing Devices and Equipment",
    "9.3 Welding Technology",
    "9.4 Weld Mechanics",
    "9.5 Tribology and Wear Protection",
    "9.6 Additive Manufacturing of Metallic Components",
    "S.1 Quality In Testing",
    "S.2 Digitalization of Quality Infrastructure",
    "S.3 Ecodesign and Energy Labelling",
    "Z.1 Organisation, Controlling",
    "Z.2 Budget",
    "Z.3 Human Resources",
    "Z.4 Education and Training, Health Management",
    "Z.5 Procurement",
    "Z.6 Internal Services",
    "Z.7 Buildings",
    "Z.8 Legal Services office, Library",
    "Z.9 Research Services"
]

### Creating the BAM organizational entity

This cell matches the organizational unit from the source dataframe against `Division_list` and stores the matching division name in `BAM Organizational Entity`.

In [ ]:
if 'Organisationseinheit' in df.columns:
    for division in Division_list:
        if division.startswith(df['Organisationseinheit'][0] + " "):
            out_df['BAM Organizational Entity'] = division          #else it is already None
            break
out_df['BAM Organizational Entity']

### Parsing the concentration value

This cell processes the concentration value from the original dataframe and stores the result in the output dataframe. Missing or non-text values are set to `None`. Percent signs and comparison symbols (`<` and `>`) are removed. The first numeric value in is extracted, converted to a number, and stored in the `Concentration` column. If conversion is unsuccessful, the value is set to `None`.

In [ ]:
if not isinstance(x, str):
    out_df['Concentration'] = None
else:
    original = df['Konzentration [%]'][0].strip()
    val = original.replace('%', '').replace('<', '').replace('>', '')

    if '-' in val:
        out_df['Concentration'] = 0.0

    else:
        match = re.search(r'[\d.,]+', val)
        if match:
            num = match.group().replace(',', '.')
            try:
                out_df['Concentration'] = float(num)
            except:
                out_df['Concentration'] = None
        else:
            out_df['Concentration'] = None
out_df['Concentration']

### Defining the notes builder

This function creates a single notes string from the selected source columns. Each available value is labeled with its column name, while empty or missing values are represented as `None`. The individual entries are joined with a separator.

In [ ]:
def build_notes(row):
    """
    Build 'Notes' by concatenating values from specific columns.
    Empty cells are replaced with 'None'.
    """
    return " | ".join(
        f"{col}: {row[col] if pd.notna(row[col]) and str(row[col]).strip() else 'None'}"
        for col in note_cols
        if col in row
    )

### Building the Notes column

This cell selects the source columns that contain additional chemical and safety information. It creates one notes string from the first dataframe row, preserving each available column name and value, and saves the result in the output dataframe's `Notes` column.

In [ ]:
note_cols = ["Piktogramme", "Reinheit", "Druck [Bar]", "max. Menge", "Konzentration [%]", "CMR", "H-Sätze", "EUH-Sätze", "P-Sätze"]
notes_string = ""

for col in note_cols:
    if col in df.columns:
        notes_string += f"{col}: {df[col].iloc[0] if pd.notna(df[col].iloc[0]) and str(df[col].iloc[0]).strip() else 'None'} | "
if len(notes_string) > 0:
    notes_string = notes_string.rstrip(" | ")

out_df['Notes'] = notes_string
out_df['Notes']

### Exporting the parsed data

The completed output dataframe is written to `output.xlsx` without the pandas index and with space reserved for the required header information. The workbook is then reopened to add the sample metadata and formatting before it is saved again.

In [ ]:
output_path = 'datasets\expected-output\output.xlsx'
out_df.to_excel(output_path, index=False, startrow=3, engine='openpyxl')

wb = load_workbook(output_path)
ws = wb.active
ws["A1"] = "SAMPLE"
ws["A2"] = "Sample type"
ws["A3"] = "CHEMICAL"
ws["A3"].font = Font(bold=True)
ws["A1"].fill = PatternFill(start_color="FFA500", end_color="FFA500", fill_type="solid")

wb.save(output_path)